# 07 — Task Decomposition and Workflow Prompting

## Scenario
Northstar receives an email requesting a refund. Our policy states that refunds are only valid if the purchase was made within the last 30 days. We have a mock database function to check purchase dates.

**The Danger:** A naive approach tries to do everything in one prompt. This forces the LLM to hallucinate database state or policy compliance because it doesn't actually have access to the deterministic data.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab07 import CASES, Draft, Extraction, build_requests, refund_eligible, run_lab, run_workflow
from northstar.contracts import check_constraints


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The "Do Everything" Baseline (Anti-Pattern)

Watch what happens when we ask the LLM to handle the whole process without giving it a way to check the actual database.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i07/naive/original-email")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = Draft.model_validate_json(response.text)
print("PARSED:", parsed)
assert check_constraints(parsed.answer, forbidden_phrases=("approved",))


## Step 2: The Sequential Workflow

We break the task down into a pipeline. 
1. **LLM Node:** Extract the Order ID.
2. **Deterministic Node:** Python checks the database.
3. **LLM Node:** Draft the response using the concrete DB fact.


In [ ]:
for case in CASES:
    extraction_request = next(r for r in build_requests() if r.case_id == f"i07/extract/{case['id']}")
    show_request(extraction_request)
    extraction_response = client.generate(extraction_request)
    extracted = Extraction.model_validate_json(extraction_response.text)
    print("RECORDED RESPONSE:", extraction_response.text)
    print("PARSED:", extracted)
    print("POLICY RESULT:", refund_eligible(extracted.order_id))
    trace = run_workflow(client, case["email"])
    print("TRACE:", trace.steps, trace.terminal_state)
    assert trace.terminal_state == case["expected_terminal"]
    if trace.terminal_state == "drafted":
        draft_request = next(r for r in build_requests() if r.case_id == f"i07/draft/{case['id']}")
        show_request(draft_request)
        draft_response = client.generate(draft_request)
        print("DRAFT RECORDED RESPONSE:", draft_response.text)
        draft = Draft.model_validate_json(draft_response.text)
        print("DRAFT PARSED:", draft)
        if refund_eligible(extracted.order_id) == "ineligible":
            assert "approved" not in draft.answer.casefold()
        else:
            assert "outside the 30-day" not in draft.answer.casefold()


## Conclusion

By decomposing the task, we prevented a hallucination, isolated the deterministic logic (the database check) from the fuzzy logic (reading/writing emails), and created observable trace points (we know exactly what Order ID was extracted).


In [ ]:
assert refund_eligible("ORD-8812") == "ineligible"
assert refund_eligible("ORD-8813") == "eligible"
assert run_workflow(client, CASES[2]["email"]).terminal_state == "clarification_required"
assert all(step["input_digest"] for step in run_workflow(client, CASES[0]["email"]).steps)


In [ ]:
results = run_lab(client)
print("POLICY VIOLATIONS METRIC:", results["policy_violations"])
print("NAIVE POLICY VIOLATIONS METRIC:", results["naive_policy_violations"])
assert results["policy_violations"].numerator == 0
assert results["naive_policy_violations"].numerator == 1


## Takeaway
The recorded workflow produced 0/2 policy violations, kept the ineligible order from an approval, and terminated the missing-ID case with `clarification_required`.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Checkpoint](README.md#checkpoint)
